# Lab 7 - Penetration Testing Basics

## Aim

- Practise early-stage penetration-testing steps in a controlled way:
  - Reconnaissance (domain/IP information).
  - HTTP header inspection.
  - Basic TCP port scanning.

> Important: these techniques must only be used on systems you own
> or have explicit permission to test.


## Reconnaissance - Domain & IP Info

In [1]:
import socket
import requests


def get_domain_info(domain: str) -> None:
    try:
        # Get IP address (active but low-risk)
        ip = socket.gethostbyname(domain)
        print(f"IP Address: {ip}")

        # Get public WHOIS-like info (passive, using a free API)
        response = requests.get(f"https://ipapi.co/{ip}/json/")
        if response.status_code == 200:
            data = response.json()
            print(f"Organization: {data.get('org', 'Unknown')}")
            print(f"City: {data.get('city', 'Unknown')}")
            print(f"Country: {data.get('country_name', 'Unknown')}")
        else:
            print("Could not fetch WHOIS data.")
    except Exception as e:
        print(f"Error: {e}")


# Example: Use a public domain (never use without permission!)
get_domain_info("python.com")

IP Address: 185.158.133.1
Could not fetch WHOIS data.


## HTTP Header Recon

In [2]:
import requests


def black_box_recon(url: str) -> None:
    try:
        response = requests.head(url)
        print("Black Box Findings:")
        print(f"Server: {response.headers.get('Server', 'Unknown')}")
        print(
            f"Content-Type: "
            f"{response.headers.get('Content-Type', 'Unknown')}"
        )
    except Exception as e:
        print(f"Error: {e}")


url = "http://python.com"
known_info = {"server": "Apache 2.4", "vulns": "Check CVE-2021-1234"}

black_box_recon(url)

Black Box Findings:
Server: cloudflare
Content-Type: Unknown


## Simple Port Scanner

In [3]:
import socket


def scan_ports(host: str, ports: list[int]) -> list[int]:
    open_ports = []

    for port in ports:
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(1)

        result = sock.connect_ex((host, port))
        if result == 0:
            open_ports.append(port)

        sock.close()

    return open_ports


host = "127.0.0.1"
ports = [80, 443, 22, 8080]

open_ports = scan_ports(host, ports)
print(f"Open ports on {host}: {open_ports}")

Open ports on 127.0.0.1: []


## Nmap scan via python-nmap

In [4]:
import nmap

def nmap_scan(host: str, port_range: str = "1-1024") -> None:
    nm = nmap.PortScanner()

    try:
        # -sV for service version detection
        nm.scan(host, port_range, arguments="-sV")

        for h in nm.all_hosts():
            print(f"Host: {h} ({nm[h].hostname()})")
            print(f"State: {nm[h].state()}")

            for proto in nm[h].all_protocols():
                print(f"Protocol: {proto}")

                lport = nm[h][proto].keys()
                for port in sorted(lport):
                    service = nm[h][proto][port]
                    print(
                        f"Port: {port}\t"
                        f"State: {service['state']}\t"
                        f"Service: {service.get('name', 'unknown')} "
                        f"{service.get('version', '')}"
                    )
    except Exception as e:
        print(f"Error: {e}")


# Example: Scan localhost
nmap_scan("127.0.0.1", "1-10")

Host: 127.0.0.1 (localhost)
State: up
Protocol: tcp
Port: 1	State: closed	Service: tcpmux 
Port: 2	State: closed	Service: compressnet 
Port: 3	State: closed	Service: compressnet 
Port: 4	State: closed	Service:  
Port: 5	State: closed	Service: rje 
Port: 6	State: closed	Service:  
Port: 7	State: closed	Service: echo 
Port: 8	State: closed	Service:  
Port: 9	State: closed	Service: discard 
Port: 10	State: closed	Service:  
